In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [ ]:
kidneyFunction = pd.read_csv(r"C:\Users\direk\Disease_risk_predictor_-3\SYSTEM\dataset\kidney_function.csv")
kidneyFunction = kidneyFunction.drop("risk_label", axis=1)
kidneyFunction.head()

,creatinine,bun,urea,uric_acid,egfr
0,1.07,15.94,34.48,5.68,124.94
1,1.04,12.20,26.09,7.13,113.25
2,1.72,24.83,53.13,4.01,52.82
3,1.06,9.91,19.42,4.04,108.25
4,1.86,22.71,48.61,5.37,46.73


In [63]:
def kidney_disease(row):
    score = 0    

    if row["egfr"] < 30:
        return "Critical"

    if row["egfr"] < 60:
        score += 2
    elif row["egfr"] < 90:
        score += 1

    if row["creatinine"] > 1.5:
        score += 1
    if row["bun"] > 20:
        score += 1
    if row["urea"] > 40:
        score += 1
    if row["uric_acid"] > 7:
        score += 1

    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"


def renal_failure(row):
    if row["egfr"] < 20 or row["creatinine"] > 2.5:
        return "Critical"
    elif row["egfr"] < 45 or row["creatinine"] > 1.8:   
        return "High"
    elif row["egfr"] < 75 or row["creatinine"] > 1.2: 
        return "Medium"
    else:
        return "Low"

In [64]:
kidneyFunction["kidney_disease"] = kidneyFunction.apply(kidney_disease, axis=1)
kidneyFunction["renal_failure"] = kidneyFunction.apply(renal_failure, axis=1)
kidneyFunction.head()

,creatinine,bun,urea,uric_acid,egfr,kidney_disease,renal_failure
0,1.07,15.94,34.48,5.68,124.94,Low,Low
1,1.04,12.20,26.09,7.13,113.25,Medium,Low
2,1.72,24.83,53.13,4.01,52.82,High,Medium
3,1.06,9.91,19.42,4.04,108.25,Low,Low
4,1.86,22.71,48.61,5.37,46.73,High,High


In [65]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [66]:
kidneyFunction["kidney_disease"] = kidneyFunction["kidney_disease"].map(LABEL_MAPPING)
kidneyFunction["renal_failure"] = kidneyFunction["renal_failure"].map(LABEL_MAPPING)
kidneyFunction.head()

,creatinine,bun,urea,uric_acid,egfr,kidney_disease,renal_failure
0,1.07,15.94,34.48,5.68,124.94,low,low
1,1.04,12.20,26.09,7.13,113.25,moderate,low
2,1.72,24.83,53.13,4.01,52.82,high,moderate
3,1.06,9.91,19.42,4.04,108.25,low,low
4,1.86,22.71,48.61,5.37,46.73,high,high


In [67]:
kidneyFunction["kidney_disease_num"] = kidneyFunction["kidney_disease"].map(NUM_MAPPING)
kidneyFunction["renal_failure_num"] = kidneyFunction["renal_failure"].map(NUM_MAPPING)
kidneyFunction.head()

,creatinine,bun,urea,uric_acid,egfr,kidney_disease,renal_failure,kidney_disease_num,renal_failure_num
0,1.07,15.94,34.48,5.68,124.94,low,low,0,0
1,1.04,12.20,26.09,7.13,113.25,moderate,low,1,0
2,1.72,24.83,53.13,4.01,52.82,high,moderate,2,1
3,1.06,9.91,19.42,4.04,108.25,low,low,0,0
4,1.86,22.71,48.61,5.37,46.73,high,high,2,2


In [68]:
"""Preparring data for ML prediction"""
feature_cols = ["creatinine", "bun", "urea", "uric_acid", "egfr"]

X = kidneyFunction[feature_cols]
y_kidney = kidneyFunction["kidney_disease_num"]
y_renal = kidneyFunction["renal_failure_num"]

In [69]:
"""train test split"""
X_train, X_test, y_train_k, y_test_k = train_test_split(
    X,
    y_kidney,
    test_size=0.2,
    random_state=31,
    stratify=y_kidney
)
_, _, y_train_r, y_test_r = train_test_split(
    X, 
    y_renal,
    test_size=0.2,
    random_state=31,
    stratify=y_renal
)

In [70]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [71]:
"""Kidney disease modal"""
model_kidney =  XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)

In [72]:
model_kidney.fit(X_train, y_train_k)
y_predict_kidney = model_kidney.predict(X_test)

In [73]:
print_report(y_test_k, y_predict_kidney, "KIDNEY DISEASE")

KIDNEY DISEASE MODEL ACCURACY
Accuracy: 99.00%

KIDNEY DISEASE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       1.00      0.98      0.99        54
    moderate       0.93      1.00      0.96        13
        high       1.00      1.00      1.00        33

    accuracy                           0.99       100
   macro avg       0.98      0.99      0.98       100
weighted avg       0.99      0.99      0.99       100



In [74]:
"""Renal failure model""" 
model_renal = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)

In [75]:
model_renal.fit(X_train, y_train_r)
y_predict_renal = model_renal.predict(X_test)

In [76]:
print_report(y_test_r, y_predict_renal, "RENAL FAILURE")

RENAL FAILURE MODEL ACCURACY
Accuracy: 92.00%

RENAL FAILURE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.98      1.00      0.99        53
    moderate       1.00      0.62      0.76        21
        high       0.79      1.00      0.88        26

    accuracy                           0.92       100
   macro avg       0.92      0.87      0.88       100
weighted avg       0.94      0.92      0.91       100



In [77]:
"""Save both models as pkl"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "kidney.pkl"), "wb") as f:
    pickle.dump(model_kidney, f)
    print("kidney.pkl saved successfully!")

with open(os.path.join(save_path, "renal.pkl"), "wb") as f:
    pickle.dump(model_renal, f)
    print("renal.pkl saved successfully!")

kidney.pkl saved successfully!
renal.pkl saved successfully!


In [78]:
# """Confusion Matrix for both models"""
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# for axes, y_test, y_pred, title in zip(
#     axes,
#     [y_test_k, y_test_r],
#     [y_predict_kidney, y_predict_renal],
#     ["Kidney Disease", "Renal Failure"]
# ):
#     existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
#     existing_names = [CLASS_NAMES[i] for i in existing_labels]